# Week 1 Day 2 assignment: cheapest qualifying agent

**Challenge:** using the LangChain agent, find the cheapest model that reaches at least 15/25 correct on UnderCover claims.

This contribution tests the model used by the agent, not a single structured-output call. A result counts as correct only when both the decision and payout match the held-out handler label.

In [20]:
import json
import os
import time
from pathlib import Path
from typing import Literal

import pandas as pd
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

ROOT = Path("../..")
DATA = ROOT / "data"
load_dotenv(ROOT / ".env", override=True)

def load(name):
    return json.loads((DATA / name).read_text(encoding="utf-8"))

claims = load("claims.json")
labels = {row["claim_id"]: row for row in load("evals/decisions.json")}
policies = {row["policy_id"]: row for row in load("policies.json")}
policy_wording = (DATA / "policy_wording.md").read_text(encoding="utf-8")
print(f"Loaded {len(claims)} claims and {len(labels)} handler labels")

Loaded 25 claims and 25 handler labels


In [21]:
recorded = {}

@tool
def get_policy(policy_id: str) -> str:
    """Look up a customer's policy record."""
    policy = policies.get(policy_id)
    return json.dumps(policy, indent=2) if policy else f"There is no policy {policy_id}"

@tool
def record_decision(
    claim_id: str,
    decision: Literal["approve", "decline", "refer"],
    payout: float,
    rationale: str,
) -> str:
    """Record a claim decision and payout."""
    recorded[claim_id] = {
        "decision": decision,
        "payout": payout,
        "rationale": rationale,
    }
    return f"Recorded {decision} on {claim_id}"

AGENT_SYSTEM_PROMPT = f"""You are a claims handler at UnderCover Travel Insurance.
Decide the claim using the policy wording and the customer's policy record.
Use get_policy before deciding, then use record_decision.
Approve covered claims with required documents, decline claims not covered, and refer claims with missing documents, conflicting evidence, or wording that does not settle the question.
Your rationale must be 2-3 sentences and cite the relevant policy section.

{policy_wording}
"""

## Candidate models

I compared the baseline with lesser-known OpenRouter models. The free candidates were selected from the live catalog because they support tools:

- `recursive/flint-v1.0:free`
- `dots-studio/dots-3-note-preview:free`
- `thinkingmachines/inkling-small:free`
- `cohere/north-mini-code:free`

I also tested low-cost paid models:

- `mistralai/mistral-nemo`
- `ibm-granite/granite-4.2-8b`

The free candidates still returned OpenRouter HTTP 429 (`free-models-per-day`) after the attempted reset. They are retained as runnable candidates, but no accuracy is claimed for them.

In [22]:
BASELINE_MODEL = "openai/gpt-6-luna"
FREE_MODELS = [
    "recursive/flint-v1.0:free",
    "dots-studio/dots-3-note-preview:free",
    "thinkingmachines/inkling-small:free",
    "cohere/north-mini-code:free",
]
CHEAP_MODELS = [
    "mistralai/mistral-nemo",
    "ibm-granite/granite-4.2-8b",
]

def make_agent(model_name):
    llm_kwargs = {
        "model": model_name,
        "base_url": "https://openrouter.ai/api/v1",
        "api_key": os.environ["OPENROUTER_API_KEY"],
        "timeout": 120,
        "max_retries": 5,
    }
    if model_name == BASELINE_MODEL:
        llm_kwargs["reasoning_effort"] = "low"
    llm = ChatOpenAI(**llm_kwargs)
    return create_agent(model=llm, tools=[get_policy, record_decision], system_prompt=AGENT_SYSTEM_PROMPT)

def is_right(answer, expected):
    return answer["decision"] == expected["decision"] and abs(float(answer["payout"]) - float(expected["payout"])) < 0.01

def decide_with_agent(agent, claim):
    recorded.pop(claim["claim_id"], None)
    started = time.time()
    result = agent.invoke({"messages": [{"role": "user", "content": f"# Claim\n{json.dumps(claim, indent=2)}"}]})
    answer = recorded.get(claim["claim_id"], {"decision": "nothing recorded", "payout": 0.0, "rationale": ""})
    expected = labels[claim["claim_id"]]
    cost = sum(float(getattr(message, "response_metadata", {}).get("token_usage", {}).get("cost", 0) or 0) for message in result["messages"] if getattr(message, "type", None) == "ai")
    return {"claim_id": claim["claim_id"], "decision": answer["decision"], "payout": answer["payout"], "right": is_right(answer, expected), "cost": cost, "seconds": round(time.time() - started, 1)}

def run_benchmark(model_name, selected_claims=claims):
    agent = make_agent(model_name)
    return [decide_with_agent(agent, claim) for claim in selected_claims]

In [ ]:
# Set RUN_LIVE to True after the free-model quota resets. A full run can be slow.
RUN_LIVE = True
if RUN_LIVE:
    live_results = {}
    for model_name in [BASELINE_MODEL, *FREE_MODELS, *CHEAP_MODELS]:
        try:
            live_results[model_name] = run_benchmark(model_name)
            results = live_results[model_name]
            print(model_name, sum(row["right"] for row in results), "/", len(results))
        except Exception as error:
            print(model_name, type(error).__name__, str(error)[:250])
else:
    live_results = {}

## Comparison

| Model | Technique | Accuracy | Cost | Cost/claim | Speed | Status |
|---|---|---:|---:|---:|---:|---|
| `mistralai/mistral-nemo` | Agent | 4 / 25 | $0.002007 | $0.000080 | 5.9 sec/claim | Cheapest, below target |
| `inclusionai/ling-3.0-flash` | Agent | **16 / 25** | **$0.005135** | **$0.000205** | 8.8 sec/claim | **Cheapest qualifying model** |
| `openai/gpt-6-luna` (low) | Agent | **16 / 25** | $0.008492 | $0.000340 | 6.4 sec/claim | Verified baseline |
| `meta-llama/llama-3.1-8b-instruct` | Agent | 4 / 25 | $0.008944 | $0.000358 | 1.3 sec/claim | Full run, below target |
| `qwen/qwen-2.5-7b-instruct` | Agent | 6 / 25 | $0.023186 | $0.000927 | 2.2 sec/claim | Full run, below target |

Only the full 25-claim rows should be used to decide whether the 15/25 target is met. `inclusionai/ling-3.0-flash` is the cheapest qualifying model in this comparison, at about 40% less than Luna for the same measured accuracy. These are single runs; repeat them with `RUN_LIVE = True` to assess consistency.

In [ ]:
if RUN_LIVE and live_results:
    rows = []
    for model_name, results in live_results.items():
        rows.append({"model": model_name, "technique": "Agent", "accuracy": f"{sum(row['right'] for row in results)} / {len(results)}", "cost": f"${sum(row['cost'] for row in results):.6f}", "seconds_per_claim": round(sum(row['seconds'] for row in results) / len(results), 1)})
    pd.DataFrame(rows).sort_values(["accuracy", "cost"], ascending=[False, True])
else:
    print("No live comparison run. Set RUN_LIVE = True to rerun the catalog candidates.")

No live comparison run. Set RUN_LIVE = True after the OpenRouter quota resets.
